In [0]:
import json
import concurrent.futures
from pyspark.sql.functions import current_timestamp, input_file_name, lit, regexp_extract, col

# ==============================================================================
# PASO 1: CONFIGURACIÓN GLOBAL Y LECTURA DE LA TABLA DE CONTROL
# ==============================================================================
catalog_name = "real_state_project"
spark.catalog.setCurrentCatalog(catalog_name)

# Ruta base donde caen TODOS los archivos CSV
base_landing_path = f"/Volumes/{catalog_name}/raw_data_real_state/real_state_csv_landing_volume/"

# En lugar de usar widgets, leemos toda la tabla de control de un solo golpe.
# (Asegúrate de poner el nombre correcto de tu tabla donde tienes la configuración)
df_control = spark.table(f"{catalog_name}.raw_data_real_state.ingestion_map") 
configuraciones = df_control.collect()

print(f"Se encontraron {len(configuraciones)} portales en la tabla de control. Iniciando proceso paralelo...")

# ==============================================================================
# PASO 2: DEFINIR LA FUNCIÓN QUE PROCESARÁ CADA PORTAL (WORKER)
# ==============================================================================
def procesar_portal(row):
    """
    Esta función contiene TU lógica exacta de ingesta y enriquecimiento, 
    pero adaptada para correr de forma aislada por cada fila de configuración.
    """
    # 2.1 Capturar valores de la fila actual (Equivalente a tus antiguos widgets)
    scraper_tool   = row["scraper_tool"]
    portal         = row["portal"]
    property_type  = row["property_type"]
    operation_type = row["operation_type"]
    file_prefix    = row["file_prefix"]
    schema_and_table = row["target_table"]
    
    # Manejo seguro del JSON de opciones de lectura
    read_options_str = row.asDict().get("read_options", "{}")
    read_options_dict = json.loads(read_options_str) if read_options_str else {}

    full_target_table = f"{catalog_name}.{schema_and_table}"
    source_path = f"{base_landing_path}{file_prefix}*"
    
    # Checkpoints y Schemas dinámicos
    checkpoint_path = f"{base_landing_path}_checkpoints/{file_prefix}"
    schema_location_path = f"{base_landing_path}_schemas/{file_prefix}"

    # ==========================================================================
    # PASO 3: ESCUDO DE PREVALIDACIÓN
    # ==========================================================================
    try:
        # Listamos los archivos en el volumen
        archivos_en_volumen = dbutils.fs.ls(base_landing_path)
        # Buscamos si hay alguno para este portal en específico
        archivos_validos = [f.name for f in archivos_en_volumen if f.name.startswith(file_prefix)]
        
        if len(archivos_validos) == 0:
            return f"⚠️ SKIP: No se encontraron archivos para '{file_prefix}'. Saltando ingesta."
    except Exception as e:
        return f"❌ SKIP: Error al validar archivos para '{file_prefix}'. Detalle: {str(e)}"

    # ==========================================================================
    # PASO 4: LECTURA INCREMENTAL Y ENRIQUECIMIENTO (TU LÓGICA)
    # ==========================================================================
    try:
        # LECTURA
        df_raw = spark.readStream \
            .format("cloudFiles") \
            .option("cloudFiles.format", "csv") \
            .option("cloudFiles.rescuedDataColumn", "_rescued_data") \
            .option("cloudFiles.schemaLocation", schema_location_path) \
            .options(**read_options_dict) \
            .load(source_path)

        # ENRIQUECIMIENTO
        df_enriched = df_raw \
            .withColumn("scraper_tool", lit(scraper_tool)) \
            .withColumn("portal", lit(portal)) \
            .withColumn("property_type", lit(property_type)) \
            .withColumn("operation_type", lit(operation_type)) \
            .withColumn("source_file", col("_metadata.file_path")) \
            .withColumn("extraction_date", regexp_extract("source_file", r"(\d{4}-\d{2}-\d{2})", 1)) \
            .withColumn("ingested_at", current_timestamp())

        # ESCRITURA EN CAPA BRONZE
        query = df_enriched.writeStream \
            .format("delta") \
            .outputMode("append") \
            .option("checkpointLocation", checkpoint_path) \
            .option("mergeSchema", "true") \
            .trigger(availableNow=True) \
            .toTable(schema_and_table)

        # Esperamos a que el micro-batch termine
        query.awaitTermination()
        
        return f"✅ ÉXITO: Ingesta completada para {full_target_table} ({len(archivos_validos)} archivos procesados)."
        
    except Exception as e:
        return f"💥 ERROR: Falló la ingesta para {file_prefix}. Detalle: {str(e)}"

# ==============================================================================
# PASO 5: EJECUCIÓN MULTIPROCESAMIENTO NATIVO CON PYTHON
# ==============================================================================
MAX_HILOS_CONCURRENTES = 10 # Procesará 10 portales a la vez

print("-" * 60)
print(f"Lanzando {MAX_HILOS_CONCURRENTES} hilos en paralelo...")

# Lanzamos la función para todas las filas de tu tabla al mismo tiempo
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_HILOS_CONCURRENTES) as executor:
    resultados = list(executor.map(procesar_portal, configuraciones))

# Imprimimos el reporte final para que lo veas en los logs del Workflow
print("\n" + "=" * 60)
print("REPORTE FINAL DE INGESTA")
print("=" * 60)
for reporte in resultados:
    print(reporte)